# TAMILAN AI — இலவச முதல் மாடல் சோதனை

இந்த notebook Google Colab-ல் இயக்குவதற்காக. இது Qwen3-1.7B-ஐ prototype candidate ஆகச் சோதிக்கும்; production server அல்ல. Colab free GPU கிடைப்பது உறுதி இல்லை. GPU கிடைக்காவிட்டால் model download செய்யாமல் நிறுத்துங்கள்.

**செலவு பாதுகாப்பு:** paid runtime / paid compute தேர்வு செய்ய வேண்டாம். எந்த API key-யும் தேவையில்லை. Model weights Google/Hugging Face-இலிருந்து பதிவிறங்கும்; அதற்கு இணையமும் நேரமும் தேவை.


In [ ]:
# 1) Runtime வகையைப் பாருங்கள். GPU இல்லாவிட்டால் இங்கேயே நிறுத்தலாம்.
import torch, os, time, json, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('GPU இல்லை. பெரிய model-ஐ download செய்ய வேண்டாம்; Colab Runtime > Change runtime type-ல் கிடைக்கும் GPU-ஐத் தேர்வு செய்து மீண்டும் இயக்கவும். Free GPU கிடைக்காமல் இருக்கலாம்.')


In [ ]:
# 2) GPU கிடைத்த பிறகே இந்த cell-ஐ இயக்கவும்.
assert torch.cuda.is_available(), 'GPU இல்லை; model download நிறுத்தப்பட்டது.'
%pip -q install -U transformers accelerate safetensors


In [ ]:
# 3) அதிகாரப்பூர்வ upstream checkpoint-ஐ load செய்கிறது.
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = 'Qwen/Qwen3-1.7B'
start = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
print('Model loaded in seconds:', round(time.time() - start, 1))
print('GPU allocated GB:', round(torch.cuda.memory_allocated() / 1024**3, 2))


In [ ]:
# 4) தமிழ் / English / Tanglish சிறிய சோதனை.
SYSTEM = ('You are TAMILAN AI, a careful multilingual career guidance assistant. '
          'Never invent live vacancies, employer facts, salary figures, or job counts. '
          'If verified live job data is not supplied, say that you cannot confirm vacancies. '
          'Do not request or reveal private personal data. Give guidance only; do not make hiring decisions.')
tests = [
    ('ta', 'நான் 12ஆம் வகுப்பு முடித்துள்ளேன். அனுபவம் இல்லை. என்ன வேலைகளைத் தேடலாம்?'),
    ('en', 'I completed a diploma and have no experience. What entry-level roles could I explore?'),
    ('ta-en-mix', 'Enakku fresher job venum, resume innum ready illa. First enna pannalaam?'),
]
def ask_tamilan_ai(question, max_new_tokens=220):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    started = time.time()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer, round(time.time() - started, 2)

for lang, question in tests:
    answer, seconds = ask_tamilan_ai(question)
    print(f'\\n--- {lang} | {seconds}s ---\\n{answer}\\n')


## முடிவுகளை எப்படிப் புரிந்துகொள்வது?

- இந்தச் சோதனை 3 prompt-களில் மட்டும் model பதிலைப் பார்க்கிறது; இது benchmark அல்லது quality approval அல்ல.
- தவறான vacancy, salary, company விவரம் உருவாக்கினால் அதை failure எனக் குறிக்கவும்.
- தமிழ் இயல்புத்தன்மை, கேள்வி புரிதல், உண்மைத் துல்லியம், பதில் நேரம் ஆகியவற்றை பதிவு செய்யவும்.
- Colab session முடிந்துவிடலாம். இதை website-ன் 24/7 backend என்று கருதக்கூடாது.
- எந்தப் பயனர் resume அல்லது தனிப்பட்ட தகவலையும் இங்கே paste செய்ய வேண்டாம்.
